**Navigation** : [Index](../../README.md)
# CC2 : OncoPlan - Squelette

**Cours :** IA101 - Intelligence Artificielle (EPF 2026)
**Étudiant(s) :** [VOTRE NOM ICI]

Ce notebook est le squelette à compléter pour le devoir OncoPlan. Il couvre :
1.  **Symbolique** : Ontologie RDF et Planification OR-Tools.
2.  **Probabiliste** : Modélisation Bayésienne avec Pyro.
3.  **Bonus** : Traçabilité via Hashage.

## Installation des Dépendances

In [1]:
# Dependances pre-provisionnees (rdflib ortools pyro-ppl torch pandas matplotlib seaborn).
# Aucun install requis : voir CaseStudies/requirements.txt ; imports directs en cellule suivante.

Import des bibliotheques necessaires.

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from rdflib import Graph, Literal, RDF, URIRef, Namespace
from rdflib.namespace import FOAF, XSD
from ortools.sat.python import cp_model
import torch
import pyro
import pyro.distributions as dist
from pyro.infer import SVI, Trace_ELBO, Predictive
from pyro.optim import Adam
import hashlib
import json

# Configuration
sns.set_style("whitegrid")
pyro.set_rng_seed(101)

---

## Partie 1 : Le Pharmacien Symbolique

### 1.1. Ontologie des Médicaments (RDFLib)

**Objectif :** Créer un graphe RDF contenant les médicaments et leurs propriétés (toxicité, incompatibilités).

In [3]:
# TODO: Créer le Namespace ONCO
# TODO: Initialiser le Graph
# TODO: Définir les classes et propriétés
# TODO: Peupler la base avec les médicaments (Cisplatine, 5-FU, Docetaxel, Gentamicine)

# VOTRE CODE ICI
pass

Verification de prescription basee sur l'ontologie.

In [4]:
def verifier_prescription(protocole_noms, patient_insuffisance_renale):
    """
    Vérifie si une prescription est valide.
    Args:
        protocole_noms (list): Liste de noms de médicaments (str)
        patient_insuffisance_renale (bool): True si le patient a une insuffisance rénale
    Returns:
        list: Liste de messages d'alerte (str)
    """
    alertes = []
    
    # TODO: Vérifier la néphrotoxicité si le patient est insuffisant rénal
    # TODO: Vérifier les incompatibilités entre paires de médicaments
    
    return alertes

# Test (Ne pas modifier)
print("--- Test 1 : Patient Sain, Protocole OK ---")
print(verifier_prescription(["Cisplatine", "5-FU"], False))

print("\n--- Test 2 : Patient Insuffisant Rénal ---")
print(verifier_prescription(["Cisplatine"], True))

print("\n--- Test 3 : Incompatibilité ---")
print(verifier_prescription(["Cisplatine", "Gentamicine"], False))

--- Test 1 : Patient Sain, Protocole OK ---
[]

--- Test 2 : Patient Insuffisant Rénal ---
[]

--- Test 3 : Incompatibilité ---
[]


### Exercice 1 : Étendre l'ontologie avec de nouveaux médicaments et interactions

**Objectif** : Ajouter de nouveaux médicaments à l'ontologie RDF et implémenter une vérification d'interactions étendue.

**Contexte** : L'ontologie actuelle contient 4 médicaments. En pratique, les protocoles oncologiques combinent davantage de molécules, et les interactions croisées sont critiques.

**Indices** :
- # Indice 1 : Ajoutez « Paclitaxel » (toxicité_rénale=False, incompatible avec « Docetaxel ») et « Carboplatine » (toxicité_rénale=True, incompatible avec « Cisplatine »)
- # Indice 2 : Vérifiez qu'un protocole `["Cisplatine", "Carboplatine"]` déclenche bien une alerte d'incompatibilité
- # Étape 1 : Ajouter les nouveaux médicaments au dictionnaire `medicaments`
- # Étape 2 : Peupler le graphe RDF avec les nouveaux triplets
- # Étape 3 : Tester la fonction `verifier_prescription` avec des combinaisons incluant les nouveaux médicaments

In [5]:
# Exercice 1 : Étendre l'ontologie médicamenteuse (voir l'énoncé ci-dessus)
# TODO étudiant : Ajouter « Paclitaxel » et « Carboplatine » au dictionnaire medicaments
# Indice : vérifier qu'un protocole ["Cisplatine", "Carboplatine"] déclenche une alerte
def etendre_ontologie(medicaments, graphe):
    """TODO étudiant : étendre l'ontologie et vérifier les interactions croisée."""
    pass  # TODO étudiant : implémenter (cf indices ci-dessus)

### 1.2. Planification des Cures (OR-Tools)

**Objectif :** Planifier 4 cycles de chimiothérapie en respectant les contraintes de temps et de ressources.

In [6]:
def planifier_chimio(nb_cycles=4, duree_cycle=21, jours_admin=[1, 8], capacite_max=3, occupations_existantes={}):
    model = cp_model.CpModel()
    horizon = nb_cycles * duree_cycle + 10 # Marge
    
    # TODO: Définir la variable de décision (Jour de début du traitement)
    
    # TODO: Ajouter les contraintes
    # 1. Pas de dimanche (supposons J1 = Lundi, donc Dimanche = 7, 14...)
    #    Indice : Utilisez AddModuloEquality pour vérifier si le jour est un multiple de 7
    # 2. Respecter la capacité max (occupations_existantes)
    
    # Résolution
    solver = cp_model.CpSolver()
    status = solver.Solve(model)
    
    if status == cp_model.OPTIMAL or status == cp_model.FEASIBLE:
        # TODO: Récupérer la valeur de début et calculer le planning complet
        debut = 0 # À remplacer
        planning = [] # À remplir
        return debut, planning
    else:
        return None, []

# Simulation d'occupations (Jours 10, 11, 12 sont complets)
occupations = {10: 3, 11: 3, 12: 3}
debut, planning = planifier_chimio(occupations_existantes=occupations)
print(f"Début optimal du traitement : Jour {debut}")
print(f"Jours d'administration : {planning}")

Début optimal du traitement : Jour 0
Jours d'administration : []


---

## Partie 2 : Le Médecin Probabiliste (Pyro)

**Objectif :** Modéliser la réponse du patient (Toxicité) en fonction des doses reçues et des observations (Globules Blancs).

In [7]:
class OncoModel:
    def __init__(self):
        pass
        
    def model(self, doses, observations=None):
        # TODO: Définir le Prior sur le profil du patient (Categorical)
        # 0: Résistant, 1: Normal, 2: Sensible
        
        # TODO: Définir la boucle temporelle
        # toxicite_cumulee évolue selon dose et sensibilité
        # mu_gb (Globules Blancs) dépend de toxicite_cumulee
        
        # TODO: Définir la Likelihood (Observation)
        pass
            
    def guide(self, doses, observations=None):
        # TODO: Définir le guide variationnel pour 'profil'
        pass

# Instanciation
onco_model = OncoModel()

# Données simulées : Patient reçoit 100mg à T0, 0 à T1... 
# Observation : Chute brutale à T1 (J8)
doses_plan = torch.tensor([100.0, 0.0, 0.0]) # J1, J8, J15
observations_reelles = torch.tensor([7800.0, 2500.0]) # J1 Normal, J8 Neutropénie sévère

# Inférence SVI (À décommenter une fois le modèle implémenté)
# pyro.clear_param_store()
# svi = SVI(onco_model.model, onco_model.guide, Adam({"lr": 0.01}), loss=Trace_ELBO())

# IMPORTANT : Pour l'inférence, assurez-vous d'aligner les doses avec les observations disponibles.
# Si vous passez des doses futures sans observations, Pyro tentera d'inférer ces observations manquantes.
# doses_inf = ... (à définir)

# print("Début de l'inférence...")
# for step in range(1000):
#    loss = svi.step(doses_inf, observations_reelles)
#    if step % 200 == 0:
#        print(f"Step {step} : Loss = {loss}")

# Résultats
# post_probs = pyro.param("profil_probs_post").detach()
# print(f"\nProbabilités a posteriori du profil :")
# print(f"Résistant : {post_probs[0]:.2f}")
# print(f"Normal    : {post_probs[1]:.2f}")
# print(f"Sensible  : {post_probs[2]:.2f}")

### 2.2. Prédiction et Prise de Décision

**Objectif :** Comparer le risque de neutropénie sévère (< 2000) pour deux scénarios futurs.

In [8]:
# TODO: Utiliser Predictive pour simuler l'avenir
# Scénario 1 : Maintien de la dose (100mg)
# Scénario 2 : Réduction de dose (50mg)

# VOTRE CODE ICI
pass

### Exercice 2 : Sensibilité du modèle bayésien au prior sur le profil patient

**Objectif** : Étudier comment le choix du prior sur les profils (Résistant/Normal/Sensible) influence le diagnostic postérieur.

**Contexte** : Le prior actuel est `[0.1, 0.6, 0.3]` (10 % Résistants, 60 % Normaux, 30 % Sensibles). Si la population est différente, le diagnostic change.

**Indices** :
- # Indice 1 : Modifiez `profil_probs` dans la méthode `model()` pour tester : `[0.33, 0.34, 0.33]` (uniforme), `[0.05, 0.90, 0.05]` (peu de sensibles)
- # Indice 2 : Relancez l'inférence SVI et comparez les probabilités a posteriori du profil
- # Étape 1 : Créer une fonction qui accepte le prior en paramètre
- # Étape 2 : Tester 3 configurations de prior différentes
- # Étape 3 : Afficher les postérieurs dans un tableau comparatif

In [9]:
# Exercice 2 : Sensibilité au prior du profil patient (voir l'énoncé ci-dessus)
# TODO étudiant : rendre profil_probs paramétrable dans OncoModel.model()
# Indice : tester [0.33, 0.34, 0.33] (uniforme) et [0.05, 0.90, 0.05] (peu de sensibles)
def model_parametrable(doses, profil_probs, observations=None):
    """TODO étudiant : version paramétrable du modèle bayésien."""
    pass  # TODO étudiant : implémenter (cf indices ci-dessus)

---

## Partie 3 : Bonus Smart Contract

**Objectif :** Garantir l'intégrité du plan de traitement.
---
**Retour au sommaire** : [Index CaseStudies](../../README.md)


In [10]:
class OncoContract:
    def __init__(self, medecin_id, patient_id):
        self.medecin_id = medecin_id
        self.patient_id = patient_id
        self.plan_hash = None
        self.signatures = {"medecin": False, "patient": False}
        
    def enregistrer_plan(self, plan_data):
        # TODO: Hasher le plan (SHA-256)
        pass
        
    def signer(self, role, cle_privee_simulee):
        # TODO: Enregistrer la signature
        pass
            
    def verifier_execution(self, plan_propose):
        # TODO: Vérifier que le hash correspond et que tout le monde a signé
        return "Non implémenté"

# Test
contract = OncoContract("DrHouse", "P004")
plan_final = {"J1": "Cisplatine 50mg", "J8": "Repos"}
contract.enregistrer_plan(plan_final)
contract.signer("medecin", "key123")
contract.signer("patient", "key456")

print(contract.verifier_execution(plan_final))
print(contract.verifier_execution({"J1": "Cisplatine 100mg"})) # Tentative de fraude

Non implémenté
Non implémenté


### Exercice 3 : Ajouter un journal d'audit au contrat intelligent

**Objectif** : Étendre la classe `OncoContract` pour maintenir un historique des modifications et détecter les tentatives de fraude.

**Contexte** : Le contrat actuel enregistre un seul plan. En pratique, les plans peuvent être modifiés plusieurs fois, et il faut tracer qui a fait quoi et quand.

**Indices** :
- # Indice 1 : Ajoutez un attribut `self.historique = []` qui stocke chaque modification
- # Indice 2 : Chaque entrée du journal doit contenir : horodatage, auteur, action, hash du plan
- # Étape 1 : Ajouter l'attribut `historique` et le mettre à jour à chaque appel de `enregistrer_plan`
- # Étape 2 : Implémenter une méthode `afficher_historique()` qui affiche le journal
- # Étape 3 : Tester en modifiant le plan plusieurs fois et vérifier que l'historique est complet

In [11]:
# Exercice 3 : Journal d'audit pour le contrat intelligent (voir l'énoncé ci-dessus)
# TODO étudiant : étendre OncoContract avec un attribut self.historique = []
# Indice : chaque entrée = (horodatage, auteur, action, hash du plan)
def creer_contrat_audite(medecin_id, patient_id):
    """TODO étudiant : retourner un OncoContract doté d'un journal d'audit."""
    pass  # TODO étudiant : implémenter (cf indices ci-dessus)